In [29]:
import numpy as np
import scipy.io
from sklearn import metrics
import pandas as pd
import os
os.environ['THEANO_FLAGS'] = "device=cuda0,force_device=True,floatX=float32"
import theano
print(theano.config.device)

from keras.layers import Embedding
from keras.models import Sequential
from keras.models import Model
from keras.layers import Dense, Dropout, Activation, Flatten, Layer, merge, Input, Concatenate, Reshape
from keras.layers.convolutional import Conv1D, MaxPooling1D
from keras.layers.pooling import GlobalMaxPooling1D
from keras.layers.recurrent import LSTM
from keras.layers.wrappers import Bidirectional, TimeDistributed
from keras.models import load_model
from keras.callbacks import ModelCheckpoint, EarlyStopping

cuda0


In [24]:
pip install --upgrade tensorflow

     |████████████████████████████████| 199.0 MB 39.3 MB/s            
  Preparing metadata (setup.py) ... done
     |████████████████████████████████| 3.9 MB 12.2 MB/s            
  Using cached keras-2.6.0-py2.py3-none-any.whl (1.3 MB)
  Created wheel for clang: filename=clang-5.0-py3-none-any.whl size=30702 sha256=972389ff896427e25b126aca55e43ebb2677dfafae8493c26db613e4cae225ed
  Stored in directory: /Users/yibeijia/Library/Caches/pip/wheels/f1/60/77/22b9b5887bd47801796a856f47650d9789c74dc3161a26d608
Successfully built clang
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.34.1
    Uninstalling grpcio-1.34.1:
      Successfully uninstalled grpcio-1.34.1
  Attempting uninstall: keras
    Found existing installation: Keras 2.4.3
    Uninstalling Keras-2.4.3:
      Successfully uninstalled Keras-2.4.3
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.5.0
    Uninstalling tensorflow-2.5.0:
      Successfully uninstalled tensorflow-2

In [26]:
pip install --upgrade pip

     |████████████████████████████████| 1.7 MB 656 kB/s            
  Attempting uninstall: pip
    Found existing installation: pip 21.3
    Uninstalling pip-21.3:
      Successfully uninstalled pip-21.3
Note: you may need to restart the kernel to use updated packages.


In [27]:
pip install --upgrade keras

Note: you may need to restart the kernel to use updated packages.


In [33]:
tensorflow.__version__


NameError: name 'tensorflow' is not defined

In [30]:
def get_auroc(preds, obs):
    fpr, tpr, thresholds  = metrics.roc_curve(obs, preds, drop_intermediate=False)
    auroc = metrics.auc(fpr,tpr)
    return auroc

def get_aupr(preds, obs):
    precision, recall, thresholds  = metrics.precision_recall_curve(obs, preds)
    aupr = metrics.auc(recall,precision)
    return aupr

def get_aurocs_and_auprs(tpreds, tobs):
    tpreds_df = pd.DataFrame(tpreds)
    tobs_df = pd.DataFrame(tobs)
    
    task_list = []
    auroc_list = []
    aupr_list = []
    for task in tpreds_df:
        pred = tpreds_df[task]
        obs = tobs_df[task]
        auroc=round(get_auroc(pred,obs),5)
        aupr = round(get_aupr(pred,obs),5)
        task_list.append(task)
        auroc_list.append(auroc)
        aupr_list.append(aupr)
    return auroc_list, aupr_list

### Load data (test)


In [31]:
# data_folder = "./data/"
data_folder = "../data/train_test_data/"
testmat = scipy.io.loadmat(data_folder+'Test_data.mat')

In [13]:
# os. getcwd()

'/Users/yibeijia/Downloads/nucleosome_occupancy/tbinet_stuff'

### Load model

In [32]:
model = load_model("./model/tbinet.h5")
print('model summary')
model.summary()

ValueError: bad marshal data (unknown type code)

### Calculate averaged AUROC and AUPR

In [ ]:
tpreds = model.predict(np.transpose(testmat['testxdata'],axes=(0,2,1)),verbose=1)
tpreds_temp = np.copy(tpreds)
reverse_start_id = int(testmat['testdata'][:,125:815].shape[0]/2)

for i in range(reverse_start_id):
    tpreds_avg_temp = (tpreds_temp[i] + tpreds_temp[reverse_start_id+i])/2.0
    tpreds_temp[i] = tpreds_avg_temp
    tpreds_temp[reverse_start_id+i] = tpreds_avg_temp


aurocs, auprs = get_aurocs_and_auprs(tpreds_temp,testmat['testdata'][:,125:815])
print("Averaged AUROC:",np.nanmean(aurocs))
print("Averaged AUPR:", np.nanmean(auprs))